In [1]:
from app.retrievers.text2cypher_builder import Text2CypherRetrieverBuilder

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.embeddings.base import Embedder
from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings
from app.agentservice import AgentService
from app.llm import LLMRegistry
from neo4j_graphrag.llm import LLMInterface
from neo4j_graphrag.llm import OpenAILLM
from app.pydantictypes import AskRequest, ClarificationRequest, MultiTurnState
from dotenv import load_dotenv
import os
import logging

# Silence Neo4j info and warning logs
logging.getLogger("neo4j").setLevel(logging.ERROR)

# 🔃 Load .env variables
load_dotenv()

# 🌍 Env vars (capitalized to distinguish)
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o")
TEMPERATURE = float(os.getenv("AZURE_OPENAI_TEMPERATURE", 0.0))
TEXT_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
LOCAL_MODE = os.getenv("LOCAL_MODE", "False")

# 🔌 Neo4j driver
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

In [5]:
import requests
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv()

# Global variables to store token and expiry
_access_token = None
_token_expiry = None

def get_access_token():
    """
    Generate a new access token using WSO2 consumer key and secret.
    This method generates the Bearer token to connect to WSO2. 
    In general, Bearer token is valid for one hour.
    """
    global _access_token, _token_expiry
    
    # Check if we have a valid cached token
    if _access_token and _token_expiry and datetime.now() < _token_expiry:
        return _access_token
    
    # Get credentials from environment variables
    client_id = os.getenv("WSO2_CONSUMER_KEY")
    client_secret = os.getenv("WSO2_CONSUMER_SECRET")
    auth_token_endpoint = os.getenv("WSO2_TOKEN_ENDPOINT")
    
    if not client_id or not client_secret:
        raise ValueError("WSO2_CONSUMER_KEY and WSO2_CONSUMER_SECRET must be set in environment variables")
    
    if not auth_token_endpoint:
        raise ValueError("WSO2_TOKEN_ENDPOINT must be set in environment variables")
    
    try:
        response = requests.post(auth_token_endpoint, data={
            'grant_type': 'client_credentials',
            'client_id': client_id,
            'client_secret': client_secret
        })
        
        if response.status_code == 200:
            token_data = response.json()
            _access_token = token_data['access_token']
            
            # Set expiry time (default to 1 hour minus configurable buffer for safety)
            expires_in = token_data.get('expires_in', 3600)  # Default to 1 hour
            buffer_seconds = int(os.getenv('TOKEN_REFRESH_BUFFER', 300))  # Default 5 minutes buffer
            _token_expiry = datetime.now() + timedelta(seconds=expires_in - buffer_seconds)
            
            print(f"✅ New access token generated, expires at: {_token_expiry}")
            return _access_token
        else:
            raise Exception(f'Failed to obtain access token: HTTP {response.status_code} - {response.text}')
    
    except Exception as e:
        print(f"❌ Error generating access token: {e}")
        raise e

def get_cached_token():
    """
    Get the currently cached token without forcing a refresh.
    Returns None if no valid token is cached.
    """
    global _access_token, _token_expiry
    
    if _access_token and _token_expiry and datetime.now() < _token_expiry:
        return _access_token
    return None

def force_token_refresh():
    """
    Force a refresh of the access token.
    """
    global _access_token, _token_expiry
    _access_token = None
    _token_expiry = None
    return get_access_token()

def get_token_expiry():
    """
    Get the current token expiry time.
    Returns None if no token is cached.
    """
    global _token_expiry
    return _token_expiry

def get_token_status():
    """
    Get comprehensive token status information.
    """
    global _access_token, _token_expiry
    
    if not _access_token:
        return {"status": "no_token", "token": None, "expiry": None, "minutes_remaining": None}
    
    if not _token_expiry:
        return {"status": "no_expiry", "token": _access_token[:30] + "...", "expiry": None, "minutes_remaining": None}
    
    now = datetime.now()
    if now >= _token_expiry:
        return {"status": "expired", "token": _access_token[:30] + "...", "expiry": _token_expiry, "minutes_remaining": 0}
    
    minutes_remaining = (_token_expiry - now).total_seconds() / 60
    return {
        "status": "valid", 
        "token": _access_token[:30] + "...", 
        "expiry": _token_expiry, 
        "minutes_remaining": round(minutes_remaining, 1)
    }

In [20]:
OPENAI_API_KEY = get_access_token()
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
llm_registry = LLMRegistry(MODEL_NAME,TEMPERATURE)

INFO:app.ws02integration.cbre_azure_chat_openai:Initializing CBRE Azure Chat OpenAI with deployment: gpt4omni


In [21]:
llm = llm_registry.langchain_llm

In [22]:
LOCAL_MODE

'False'

In [23]:
retriever = Text2CypherRetrieverBuilder(driver=driver,database='neo4j',llm=llm).build()

ERROR:neo4j.io:[#DA8D]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-d2c6a330-df1c-0001.production-orch-0451.neo4j.io', 7687)) (ResolvedIPv4Address(('10.184.80.70', 7687))): OSError('No data')


✅ Loaded 16 query examples from query_examples.yml
📊 Schema loaded: 12645 characters
📝 Examples loaded: 16 examples

🔎 Text2CypherRetriever Summary
────────────────────────────────────────
🔤 LLM type: <class 'app.ws02integration.cbre_azure_chat_openai.CBREAzureChatOpenAI'>
🧠 LLM model name: None
📁 Examples file: query_examples.yml
📝 Examples count: 16

📜 Neo4j Schema Snippet:
Node properties:
Capability {policy: STRING, object: STRING, limit_property_name: STRING, limit_name: STRING, incomplete_timeout: INTEGER, graph_limit_property: INTEGER, ttl: INTEGER, inUse: BOOLEAN, name: STRING, id: STRING, fact_property_projected_cost_currency: STRING, fact_property_current_cost_currency: STRING, fact_property_current_cost: STRING, fact_property_projected_cost: STRING, fact_currency_property_name: STRING, fact_property_name: STRING, is_external: BOOLEAN}
Structure {friendly_key: STRING, name: STRING, id: STRING, status: STRING, graph_object_id: STRING}
Object {references_structure: STRING, type

In [24]:
type(llm)

app.ws02integration.cbre_azure_chat_openai.CBREAzureChatOpenAI

In [25]:
llm.get_authentication_status()

{'status': 'no_token',
 'token': None,
 'expiry': None,
 'minutes_remaining': None}

In [26]:
llm.invoke("what is the capital of France?")

✅ New access token generated, expires at: 2025-08-19 13:15:19.229241


INFO:httpx:HTTP Request: POST https://api-test.cbre.com/t/digitaltech_us_edp/cbreopenaiendpoint/1/openai/deployments/gpt4omni/chat/completions?api-version=2024-02-15-preview "HTTP/1.1 200 OK"


AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-C6KrmNv44nLAB6cNfsDzU2S5lN7pR', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sex

In [28]:
output = retriever.get_search_results("how many nodes are there in my database")


INFO:httpx:HTTP Request: POST https://api-test.cbre.com/t/digitaltech_us_edp/cbreopenaiendpoint/1/openai/deployments/gpt4omni/chat/completions?api-version=2024-02-15-preview "HTTP/1.1 200 OK"


In [29]:
print(output)

records=[<Record node_count=5149270>] metadata={'cypher': 'MATCH (n)\nRETURN count(n) AS node_count'}
